In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

from churn import config


In [3]:
df=pd.read_parquet(config.CLEAN_DATA_FILE)
X = df.drop(columns=[config.TARGET])
y = df[config.TARGET]

Train-test-split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [5]:
print(f"X_train.shape: {X_train.shape}, X_test.shape: {X_test.shape}")
print(f"churn rate  train={y_train.mean():.4f}  test={y_test.mean():.4f}")

X_train.shape: (5634, 20), X_test.shape: (1409, 20)
churn rate  train=0.2654  test=0.2654


In [6]:
X_train.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
3738,4950-BDEUX,Male,No,No,No,35,No,No phone service,DSL,No,No,Yes,No,Yes,Yes,Month-to-month,No,Electronic check,49.20,1701.65
3151,7993-NQLJE,Male,No,Yes,Yes,15,Yes,No,Fiber optic,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,75.10,1151.55
4860,7321-ZNSLA,Male,No,Yes,Yes,13,No,No phone service,DSL,Yes,Yes,No,Yes,No,No,Two year,No,Mailed check,40.55,590.35
3867,4922-CVPDX,Female,No,Yes,No,26,Yes,No,DSL,No,Yes,Yes,No,Yes,Yes,Two year,Yes,Credit card (automatic),73.50,1905.70
3810,2903-YYTBW,Male,No,Yes,Yes,1,Yes,No,DSL,No,No,No,No,No,No,Month-to-month,No,Electronic check,44.55,44.55


Pipeline

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import FunctionTransformer

from churn.features import collapse_no_internet, build_preprocessor, add_features

In [8]:
def build_pipeline(model, model_type, num_cols=None, cat_cols=None):
    """Raw rows in → churn prediction out. model_type: "lr" or "tree" (picks the preprocessor)."""
    return Pipeline([
        ("collapse", FunctionTransformer(collapse_no_internet)),
        ("features", FunctionTransformer(add_features)),
        ("prep", build_preprocessor(model_type, num_cols, cat_cols)),
        ("model", model),
    ])

In [9]:
pipe_lr = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr").fit(X_train, y_train)

In [10]:
pipe_lr[:-1].transform(X_train.head())

,num__tenure,num__MonthlyCharges,num__num_services,cat__SeniorCitizen_Yes,cat__Partner_Yes,cat__Dependents_Yes,cat__MultipleLines_No,cat__MultipleLines_No phone service,cat__MultipleLines_Yes,cat__InternetService_DSL,...,cat__PaymentMethod_Bank transfer (automatic),cat__PaymentMethod_Credit card (automatic),cat__PaymentMethod_Electronic check,cat__PaymentMethod_Mailed check,cat__tenure_group_0-6,cat__tenure_group_13-24,cat__tenure_group_25-48,cat__tenure_group_49-72,cat__tenure_group_7-12,cat__has_family_Yes
3738,0.102371,-0.521976,0.214400,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3151,-0.711743,0.337478,-0.429486,0.0,1.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
4860,-0.793155,-0.809013,0.858287,0.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0
3867,-0.263980,0.284384,0.858287,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3810,-1.281624,-0.676279,-1.073373,0.0,1.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate

In [13]:
# Define metrics relevant for imbalanced data
scoring_metrics = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision"
}

In [14]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs = -1)
cv_lr_df = pd.DataFrame(cv_lr)
cv_lr_df

,fit_time,score_time,test_f1,test_precision,test_recall,test_roc_auc,test_pr_auc
0,0.194252,0.096997,0.587361,0.661088,0.528428,0.844498,0.645928
1,0.180145,0.074283,0.549242,0.633188,0.484950,0.829702,0.652839
2,0.194106,0.082731,0.610801,0.689076,0.548495,0.843165,0.681175
3,0.138584,0.084240,0.625000,0.670498,0.585284,0.859457,0.662347
4,0.186493,0.076125,0.615679,0.718750,0.538462,0.860106,0.675849


#### Notes 1: how to read `cv_lr_df` (Logistic Regression, 5-fold CV)

**The table**
- **1 row = 1 fold.** Each fold trains the whole Pipeline on 4/5 of `X_train` and scores the remaining 1/5 (≈ 1,056 customers).
- **`test_*` = the validation fold**, *not* `X_test`. That's sklearn's naming; the real test set stays untouched.
- `fit_time` / `score_time`: seconds to train / to predict + score. Useful later to compare slow vs fast models.

**`.agg(["mean", "std"]).T`**
- **mean** = the score to expect on new customers. Use it to **compare models**.
- **std** = how much the score moves between folds, i.e. how **stable** (trustworthy) the mean is.
- Rule of thumb: if two models' means differ by **less than about one std**, the difference may just be noise from how the folds were split. Don't call a winner on it.

**Is a small std good?** Yes. It means the score doesn't depend on which customers ended up in which fold. For ≈ 1,000 rows per fold:

| std | Reading |
|---|---|
| < 0.02 | Very stable |
| 0.02 – 0.05 | Normal for this data size (our P, R, F1, PR-AUC ≈ 0.03–0.05) |
| > 0.05 | Unstable: check the data or the model before trusting the mean |

**What counts as a good score?**

| Metric | Random model | Our LR | Good on this dataset | Warning sign |
|---|---|---|---|---|
| ROC-AUC | 0.50 | **0.85 ± 0.02** | 0.83 – 0.86 (typical tuned models) | > 0.90 → suspect leakage |
| PR-AUC | 0.265 (= churn rate) | **0.66 ± 0.05** | 0.60 – 0.70 | Near 0.265 → model learned nothing |
| Precision / Recall / F1 | depends on threshold | 0.68 / 0.53 / 0.59 | No fixed range: they change with the threshold (phase 6) | n/a |

ROC-AUC and PR-AUC don't use a cut-off, so they get "good ranges". Precision, recall and F1 here use the default 0.5 cut-off, so they are only a first look.


In [15]:
cv_lr_df.agg(["mean", "std"]).T   # .T turns it so each metric is a row


,mean,std
fit_time,0.178716,0.023191
score_time,0.082875,0.008954
test_f1,0.597617,0.030394
test_precision,0.674520,0.031910
test_recall,0.537124,0.036222
test_roc_auc,0.847386,0.012712
test_pr_auc,0.663628,0.014904


### Notes 2: what the mean scores mean for the business

**The confusion matrix words** (positive = churn = 1):

| | Model says **churn** (predicted 1, flagged) | Model says **stays** (predicted 0) |
|---|---|---|
| **Really churns** (actual 1) | **TP**: caught, gets an offer ✅ | **FN**: missed, leaves → *lost revenue* (expensive) |
| **Really stays** (actual 0) | **FP**: false alarm → *wasted offer* (cheaper) | **TN**: correctly left alone |

**One validation fold in numbers** (≈ 1,056 customers, ≈ 280 real churners), with the mean scores at the 0.5 cut-off:

| Metric | Formula | Mean | In plain words | In one fold |
|---|---|---|---|---|
| **Precision** | TP / (TP + FP) | 0.68 | Of the customers we **flag**, 68% really churn | ≈ 218 flagged → 148 real churners, **70 wasted offers** |
| **Recall** | TP / (TP + FN) | 0.53 | Of all real **churners**, we catch 53% | 148 caught, **132 missed** (lost revenue) |
| **F1** | 2·P·R / (P + R) | 0.59 | One number that balances P and R; low if either one is low | n/a |
| **ROC-AUC** | P(random churner scored higher than random stayer) | 0.85 | The model **ranks** churners above stayers 85% of the time | Uses probabilities, not the cut-off |
| **PR-AUC** | Precision averaged over all recall levels | 0.66 | How good the flagged list is at every list length. Random = 0.265 | ≈ 2.5× better than random |

**Business reading**
- **Recall gets extra weight:** a missed churner (FN) costs more than a wasted offer (FP), but precision keeps the budget and team workload in check. Phase 6 balances the two by cost.
- **Precision protects the budget and the team's time.** It says how many offers are wasted, and the number flagged must fit the retention team's capacity.
- **Precision, recall and F1 change with the threshold.** Lowering the cut-off from 0.5 flags more customers: recall ↑, precision ↓. Phase 6 picks the threshold from the cost of FN vs FP. So, for now, **don't compare models on these three**.
- **ROC-AUC and PR-AUC don't depend on the threshold.** They measure how well the model *orders* customers by risk, which is what the retention team uses ("call the top N"). **Models are ranked on these two in phase 5.**
- **Accuracy is left out on purpose.** Predicting "nobody churns" already scores 73.5% but catches 0 churners.

**Why recall gets more weight than precision: the two mistakes cost different amounts**

| Metric | The mistake it counts | What that mistake costs |
|---|---|---|
| **Recall** = TP / (TP + **FN**) | **FN**: a churner we didn't flag | They leave. We lose their future monthly revenue. |
| **Precision** = TP / (TP + **FP**) | **FP**: a loyal customer we flagged | We give a discount to someone who would have stayed. We lose the cost of one offer. |

> ⚠️ **The dollar figures below are MADE-UP numbers**, only to show the idea. The dataset has no cost data, and no spec gives these values: phase 1 defines only the *formulas*, and the real numbers are chosen and justified in **phase 6, step T1**. Only the ~$75/month is loosely based on the data (≈ average `MonthlyCharges` of churners); **12 months, 30% and $50 are invented.**

- **FN (missed churner):** lost revenue ≈ `MonthlyCharges` × months they'd have stayed × chance the offer would have kept them.
  - Made-up example: $75/month × 12 months × 30% success ≈ **$270**.
- **FP (wasted offer):** one discount; made-up example **$50**.

With these made-up numbers, **one missed churner costs about 5× more than one wasted offer**. Trading one FN for a few FPs is then still a profit, which is why the business leans towards **catching more churners (recall)**, even if a few more loyal customers get a discount (lower precision). Step T7 checks how much the threshold moves if the real ratio is 3:1 or 10:1 instead.


### Notes 3: ROC-AUC and PR-AUC (the two metrics we rank models on)

Both start from the model's **churn probability** (`predict_proba`), not from a yes/no prediction. Picture the model sorting all customers into a **risk list**, highest risk at the top. Every possible threshold is a cut-off point in that list: everyone above it gets an offer. Both curves show what happens at **every** cut-off, so they judge the **ranking**, not one threshold.

---

#### 1. Full forms and definitions

| | **ROC-AUC** | **PR-AUC** (sklearn: `average_precision`) |
|---|---|---|
| Full form | **R**eceiver **O**perating **C**haracteristic, **A**rea **U**nder the **C**urve | **P**recision–**R**ecall, **A**rea **U**nder the **C**urve |
| Plots | **TPR** (y) vs **FPR** (x) | **Precision** (y) vs **Recall** (x) |
| Formulas | TPR = TP / (TP + FN) = **recall**<br>FPR = FP / (FP + TN) | Precision = TP / (TP + FP)<br>Recall = TP / (TP + FN) |
| Definition | Probability that a **random churner** gets a **higher** risk score than a **random loyal customer** | Average precision over all recall levels: how "clean" the call list stays as we go further down it |
| Random model | **0.50** | **0.265** (= the churn rate) |
| Perfect model | 1.0 | 1.0 |
| Our LR (CV) | **0.85 ± 0.02** | **0.66 ± 0.05** |

**In Telco business words**
- **TPR (recall):** share of churners we reach with an offer. *Revenue saved.*
- **FPR:** share of **loyal** customers who get an offer they didn't need. *Wasted discounts.*
- **Precision:** share of the **call list** that really churns. *How many of the team's calls are worth making.*
- **ROC-AUC 0.85:** pick one customer who churned and one who stayed; 85% of the time the model puts the churner higher on the risk list.
- **PR-AUC 0.66:** as the retention team works down the list, on average about 2 of every 3 calls reach a real churner (random calling: about 1 in 4).

---

#### 2. What the charts look like

**ROC curve**
```
TPR   1.0 |        ____-----‾‾‾‾
(recall)  |    _-‾‾          .·
          |  /'           .·      ← diagonal = random guessing (AUC 0.5)
          | /   ← ours  .·
          |/  (AUC .85).·
      0.0 +·------------------
          0.0      FPR       1.0
```
- Each point on the curve is **one threshold**. Bottom-left: threshold 1.0, nobody flagged. Top-right: threshold 0, everyone flagged.
- **Closer to the top-left corner = better:** we catch many churners (high TPR) while bothering few loyal customers (low FPR).
- Our 0.5 cut-off is one point: about (FPR 0.09, TPR 0.53) in a fold. 70 of ≈ 776 loyal customers flagged, 148 of ≈ 280 churners caught.  
- (FPR 0.09, TPR 0.53) => It catches about 53% of churners while bothering about 9% of loyal customers.

**PR curve**
```
Precision 1.0 |‾‾\_
              |    ‾\__          ← ours (AUC .66)
              |        ‾‾\__
              |             ‾\_
        0.265 |- - - - - - - - -‾‾  ← random = churn rate (flat line)
              +-------------------
              0.0     Recall     1.0
```
- Read it **left to right as the call list getting longer**. On the left, only the top-risk customers are called and precision is high. Calling more customers raises recall, but more of the calls are wasted, so precision falls.
- **Higher and flatter = better:** the list stays "clean" for longer.
- At recall 1.0 (call everyone) precision drops to 0.265, because then the list is just the whole customer base.

**Why look at both?** Churners are the minority (26.5%). On the ROC chart, 70 wasted offers look small (FPR = 70 / 776 ≈ 9%), because the denominator is all loyal customers. On the PR chart, the same 70 are 70 of the 218 calls (≈ 32% wasted), which is what the retention team actually feels. **ROC-AUC = overall ranking skill; PR-AUC = quality of the call list.**



#### 3. Using them to compare models in CV and pick the best

**a) The numbers (what the comparison table uses)**
1. For each model, run `cross_validate` and take `test_roc_auc` and `test_pr_auc` as **mean ± std** over the 5 folds.
2. **Rank by mean ROC-AUC**; use **PR-AUC as the tie-breaker** (the spec's rule).
3. **Gap smaller than about one std → treat it as a tie.** Prefer the simpler, more explainable model (e.g. LR over XGBoost). If LR is within ~0.01 of the best boosted model, say so openly.
4. **Check overfitting:** with `return_train_score=True`, train ROC-AUC − CV ROC-AUC should be < ~0.05 (leakage check L6).
5. **Sanity check:** CV ROC-AUC > 0.90 on this dataset → suspect leakage, not a great model.

**b) The curves (the picture behind the numbers)**
- To draw CV curves without touching `X_test`, get **out-of-fold probabilities** on the training set: `cross_val_predict(pipe, X_train, y_train, cv=skf, method="predict_proba")[:, 1]`. Each customer is scored by a model that never saw them.
- Plot every model's ROC curve on one chart and every PR curve on another (`RocCurveDisplay.from_predictions`, `PrecisionRecallDisplay.from_predictions`).
- **One curve above another everywhere** → that model is better at every threshold. Easy choice.
- **Curves cross** → look at the region the business will use. If the team can only call the top ~20% of customers, compare the curves at **low FPR / low recall** (the left side), not the whole area.

**c) Business link**
- AUC picks the model that **ranks** churners best. That's the right job for phase 5, because the threshold isn't chosen yet.
- In **phase 6**, the best model's curve is where we pick **one point** (the threshold) by cost: cost per missed churner (FN) vs cost per wasted offer (FP), within the team's capacity. *(The $270 vs $50 used in Notes 2 are **made-up numbers**; the real costs are set in phase 6, step T1.)*


#### 4. `"roc_auc"` vs `"average_precision"` in `cross_validate`: what each value in `cv_lr_df` is

Both scorers call **`predict_proba()`**, so neither uses the 0.5 threshold. In each fold, sklearn takes the validation customers' churn probabilities, tries **every** threshold, builds a curve, and turns the curve into one number. One number per fold gives the 5 rows; their mean is the value we report.

| | `"roc_auc"` → `test_roc_auc` | `"average_precision"` → `test_pr_auc` |
|---|---|---|
| Curve built per fold | ROC curve: TPR vs FPR | Precision–recall curve: precision vs recall |
| How the area is measured | **Exact area** under the ROC curve | **Staircase sum:** AP = Σ (Rₙ − Rₙ₋₁) × Pₙ. At each threshold: how much recall went up × the precision there, all added up |
| Is it "the AUC"? | Yes, it is the ROC-AUC | It is the standard way to report **PR-AUC**. Very close to the area under the PR curve, but uses rectangles (steps) instead of straight lines between points |
| Random model | 0.50 | 0.265 (= churn rate) |
| Our LR, mean of 5 folds | **0.846** | **0.663** |
| Business question it answers | "Does the model rank churners above loyal customers?" | "How clean is the call list as the team works down it?" |

**Why `average_precision` and not `auc(recall, precision)`?**
- `sklearn.metrics.auc(recall, precision)` joins the PR points with **straight lines** (trapezoids). On a PR curve that can **over-estimate** the area.
- `average_precision` uses **steps**, so it doesn't over-estimate. It is sklearn's recommended PR-AUC.
- The two numbers are usually very close; just say which one you report.

**Interview phrasing:** "I report PR-AUC as average precision, the step-wise area under the precision–recall curve, because trapezoidal interpolation can over-estimate it."


### Adding Training Score to CV

We also want the score on the **training folds**: customers the model **learned from**. That's like testing the student with the **same** questions they practised on.

Comparing the two tells you something important:

| Training score vs validation score | What it means | Student analogy |
| --- | --- | --- |
| About the same | The model learned real patterns ✅ | Understood the topic |
| Training much higher | The model **memorised** the training customers (overfitting) ❌ | Memorised the answers, fails on new questions |

This is leakage check **L6**. The spec says the gap should be **below about 0.05**.

In [16]:
cv_lr = cross_validate(pipe_lr, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1,
                       return_train_score=True)   # ← new: also score the training folds

cv_lr_df = pd.DataFrame(cv_lr)
cv_lr_df.agg(["mean", "std"]).T   # .T turns it so each metric is a row

,mean,std
fit_time,0.141400,0.005975
score_time,0.084559,0.012631
test_f1,0.597617,0.030394
train_f1,0.599111,0.009545
test_precision,0.674520,0.031910
train_precision,0.678548,0.007250
test_recall,0.537124,0.036222
train_recall,0.536455,0.013860
test_roc_auc,0.847386,0.012712
train_roc_auc,0.852222,0.003012



|  | Value |
| --- | --- |
| Training ROC-AUC (customers it learned from) | 0.852 |
| Validation ROC-AUC (customers it hasn't seen) | 0.847 |
| **Gap** | **0.005**, far below the 0.05 limit ✅ |

The gap is **0.852 - 0.847 = 0.005**.

### What it means

* The model scores **almost the same** on new customers as on the ones it learned from. In the student analogy, it **understood** the topic rather than memorising answers. **No overfitting.**


* That's expected for Logistic Regression. It's a simple model (one weight per feature), so it doesn't have enough "memory" to memorise 4,500 customers.


* Something to keep in mind for later: when train and validation are close **and** both are moderate, the model is limited by its **simplicity**, not by overfitting. That's why the spec goes on to try more flexible models (trees, boosting). They might find patterns LR can't, but they're also the ones that tend to overfit. Then this same gap check will matter.



**Interview phrasing:** "Logistic Regression had a train–CV ROC-AUC gap of 0.005, so it wasn't overfitting. Its limit is model simplicity, which is why I compared more flexible models next."

Creating function `run_cv` to be added in `Churn/`

In [17]:
def run_cv(pipe, name):
    """Run 5-fold CV on a pipeline; return one summary row for the comparison table."""
    # Define metrics relevant for imbalanced data
    scoring_metrics = {
        "f1": "f1",
        "precision": "precision",
        "recall": "recall",
        "roc_auc": "roc_auc",
        "pr_auc": "average_precision"
    }
    
    ## Run cross-validation
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv = pd.DataFrame(cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1, return_train_score=True))

    ### build the summary as a dictionary for model
    row = {
        "model": name,
        "roc_auc": cv['test_roc_auc'].mean(),          # mean of the "test_roc_auc" column
        "roc_auc_std": cv['test_roc_auc'].std(),      # std of the same column
        "pr_auc": cv['test_pr_auc'].mean(),           # mean of "test_pr_auc"
        "pr_auc_std": cv['test_pr_auc'].std(),
        "precision": cv['test_precision'].mean(),        # mean of "test_precision"
        "recall": cv['test_recall'].mean(),           # mean of "test_recall"
        "f1": cv['test_f1'].mean(),               # mean of "test_f1"
        "gap_roc_auc": cv['train_roc_auc'].mean() - cv['test_roc_auc'].mean(),      # mean of train_roc_auc minus mean of test_roc_auc
        "fit_time": cv['fit_time'].mean(),         # mean of "fit_time"
    }
    return row


In [18]:
run_cv(pipe_lr, "Logistic Regression")

{'model': 'Logistic Regression',
 'roc_auc': np.float64(0.8473855457846264),
 'roc_auc_std': np.float64(0.012712113453713656),
 'pr_auc': np.float64(0.6636277033374488),
 'pr_auc_std': np.float64(0.01490444277045332),
 'precision': np.float64(0.674519870715568),
 'recall': np.float64(0.5371237458193979),
 'f1': np.float64(0.5976165080415102),
 'gap_roc_auc': np.float64(0.004836703347998239),
 'fit_time': np.float64(0.1473027229309082)}

### Data leakage Checks

##### L2 Leakage check

In [21]:
y_train.value_counts(normalize=True)   # check the churn rate in the training set

Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

In [20]:
y_test.value_counts(normalize=True)   # check the churn rate in the training set

Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64

I split first, stratified on churn, and confirmed train and test both have ≈ 26.5% churners. So the test set is untouched and representative, and CV and test scores are comparable.

#### L1: Target Leakage Review

In [ ]:
df.loc[df['Churn']==1, ['tenure',  'MonthlyCharges', 'TotalCharges','Churn']].head(10)

,tenure,MonthlyCharges,TotalCharges,Churn
2,2,53.85,108.15,1
4,2,70.70,151.65,1
5,8,99.65,820.50,1
8,28,104.80,3046.05,1
13,49,103.70,5036.30,1
...,...,...,...,...
7021,12,59.80,727.80,1
7026,9,44.20,403.35,1
7032,1,75.75,75.75,1
7034,67,102.95,6886.25,1


* 🟩 **L1 passed.** Your conclusion is right: every feature is known at scoring time, so nothing leaks.



One thing to add for the **engineered** features. They're safe for two reasons:

1. They're **built only from known columns**: `tenure_group` $\leftarrow$ `tenure`, `num_services` $\leftarrow$ the 5 add-ons, `has_family` $\leftarrow$ Partner + Dependents.


2. They **don't learn from the data or the target**. The tenure bins are fixed edges from config, not quantiles, and none of them looks at `Churn`.



**Write it down in the notebook.** L1 says "record: the table in the notebook". A grouped table is enough, so you don't need 18 rows:

| Feature group | Columns | Known before churn? | Note |
| --- | --- | --- | --- |
| Account | tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges | Yes | Current values available in billing at scoring time. Tenure is frozen at exit for churners (snapshot caveat, not leakage) |
| Demographics | SeniorCitizen, Partner, Dependents | Yes | Known at sign-up |
| Services | MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV | Yes | Current subscriptions |
| Engineered | tenure_group, num_services, has_family | Yes | Built only from the rows above; fixed rules, no target used |
| Excluded | customerID, TotalCharges, gender, PhoneService, StreamingMovies | n/a | See `config.DROP_COLS` |

So far, L1 and L2 have both passed.

### L3 Duplicates Check

In [40]:
train_profiles = X_train.drop(columns=config.ID_COL)
test_profiles  = X_test.drop(columns=config.ID_COL)

shared = pd.merge(train_profiles, test_profiles, how="inner").drop_duplicates()
## shared only has rows which are common to both the training and test sets. 
## This is a problem because it means that the model has already seen these rows during training, which can lead to data leakage and overfitting. 
## We need to ensure that the training and test sets are completely separate and do not share any rows.
len(shared)


12

22 feature profiles repeat across different customers; 12 of them span the train/test split. That's under 1% of test, they're genuine customers, so I kept them and recorded the count rather than deleting real data.

### L5 the single-feature AUC scan (the last check before modelling)

**The idea.** L1 checked **timing** by reasoning. L5 checks **strength** with numbers. For each feature on its own, train a tiny model that uses **only that one column** and get its CV ROC-AUC.

* $\text{AUC} \approx 0.5 \rightarrow$ this feature alone is useless.


* $\text{AUC} \approx 0.6\text{--}0.8 \rightarrow$ a normal, useful feature.


* **$\text{AUC} > 0.90$ from ONE column** $\rightarrow$ suspicious. Real churn is never that easy to predict from a single fact, so you investigate it as possible leakage.

**Bonus:** the ranking should match your EDA. Contract and tenure should be near the top, and that's a nice "my EDA and my model agree" moment.

In [43]:
len(config.NUM_COLS + config.CAT_COLS)

18

In [46]:
rows = []
for col in config.NUM_COLS + config.CAT_COLS:
    if col in config.NUM_COLS:
        pipe = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", num_cols=[col], cat_cols=[])
    else:
        pipe = build_pipeline(LogisticRegression(max_iter=1000), model_type="lr", num_cols=[], cat_cols=[col])

    cv = run_cv(pipe, "Logistic Regression")
    rows.append({"Column": col, "mean_roc_auc": cv["roc_auc"], "pr_auc": cv["pr_auc"]})

auc_scan = pd.DataFrame(rows).sort_values(by="mean_roc_auc", ascending=False)
auc_scan


,Column,mean_roc_auc,pr_auc
13,Contract,0.739499,0.415703
0,tenure,0.735243,0.478918
16,tenure_group,0.720872,0.437146
7,InternetService,0.697570,0.387264
15,PaymentMethod,0.676426,0.394965
1,MonthlyCharges,0.623485,0.316684
14,PaperlessBilling,0.610222,0.320356
17,has_family,0.589851,0.310492
8,OnlineSecurity,0.589535,0.306816
5,Dependents,0.586739,0.305367


Good question, and the answer is short. Your `run_cv` already returns **both**, so keep both columns in the table. ROC-AUC is the one we **sort and judge by** in this check, for one reason: **its scale is the same on every dataset.**

|  | Random-guess value | "Suspiciously high" |
| --- | --- | --- |
| **ROC-AUC** | **0.5** always, whatever the churn rate | $> 0.90$ is a widely used alarm level |
| **PR-AUC** | **= the churn rate** ($0.265$ here; $0.05$ on a $5\%$ churn dataset) | no universal number; it depends on the dataset |

L5 is an **alarm with a fixed rule** ("any single feature $> 0.90 \rightarrow$ investigate"). A fixed rule needs a metric whose "random" and "perfect" points don't move. ROC-AUC has that, and PR-AUC doesn't. A PR-AUC of $0.50$ means very different things at $26.5\%$ churn and at $5\%$ churn.

**When PR-AUC is the better choice:** comparing **models on the same dataset** for the business decision, because it focuses on how well we find the minority churners. That's why the spec uses it as the tie-breaker in step 10. Here, though, we're not choosing a model, just looking for a cheating feature.

**In practice:** sort by `roc_auc`, keep `pr_auc` alongside, and glance at whether both rank the same features at the top. They usually agree. If one feature were high on PR-AUC but ordinary on ROC-AUC, that would also be worth a look.

> As a leakage check I scanned each feature's standalone CV ROC-AUC. The max was Contract at 0.74, nowhere near the 0.90 alarm, and the ranking matched my EDA: contract and tenure on top.